In [13]:
pip install neo4j pandas fuzzywuzzy python-levenshtein openpyxl

Note: you may need to restart the kernel to use updated packages.


In [20]:
import os
import re
import glob
import pickle
import numpy as np
from neo4j import GraphDatabase
from langchain_huggingface import HuggingFaceEmbeddings
from sklearn.metrics.pairwise import cosine_similarity

# --- CONFIGURATION ---
NEO4J_URI = "neo4j://127.0.0.1:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "kg123456" 
VECTORS_PATH = "kg_vectors.npy"
NAMES_PATH = "kg_names.pkl"
HF_HUB_PATH = r"D:\huggingface\hub\models--FremyCompany--BioLORD-2023\snapshots"

# --- 3.2.2 ENTITY TYPE WEIGHTS (As defined in medical logic) ---
TYPE_WEIGHTS = {
    "Symptom": 1.0,    # sym - Direct clinical signal
    "Microbe": 0.8,    # mic - Pathogenic cause
    "LabItem": 0.7,    # ite - Diagnostic marker
    "Drug": 0.5,       # dru - Interventional link
    "Procedure": 0.3,  # pro - Diagnostic procedure
    "BodyPart": 0.2    # bod - Anatomical grounding
}

class BioLORDMatcher:
    """Implements Section 3.2.1: Entity Recognition and Matching"""
    def __init__(self):
        snapshots = glob.glob(os.path.join(HF_HUB_PATH, "*"))
        self.embeddings_model = HuggingFaceEmbeddings(
            model_name=snapshots[0],
            model_kwargs={'device': 'cpu', 'local_files_only': True}
        )
        self.kg_vectors = np.load(VECTORS_PATH)
        with open(NAMES_PATH, 'rb') as f: self.kg_names = pickle.load(f)

    def match(self, raw_term: str, threshold=0.60):
        query_vec = np.array(self.embeddings_model.embed_query(raw_term.lower())).reshape(1, -1)
        sims = cosine_similarity(query_vec, self.kg_vectors)[0]
        idx = np.argmax(sims) # eq 4: arg max sim
        return self.kg_names[idx] if sims[idx] >= threshold else None

class PaperKGAgent:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def close(self):
        self.driver.close()

    def generate_candidates(self, resolved_entities: list):
        if not resolved_entities: return []
        entity_names = [e['name'] for e in resolved_entities]
        
        with self.driver.session() as session:
            query = """
            MATCH (d) WHERE any(lb IN labels(d) WHERE lb IN ['Disease', 'dis'])
            
            // 1. Filter out demographic/lab noise
            WITH d, $input_entities AS inputs
            WHERE NOT d.name CONTAINS 'Neonat' 
              AND NOT d.name CONTAINS 'placenta'
              AND NOT d.name CONTAINS 'NB '
              AND NOT d.name CONTAINS 'high'
              AND NOT d.name CONTAINS 'low'

            // 2. Find Paths (Using 4 hops to ensure we reach 'Fever' and 'Chest Pain')
            OPTIONAL MATCH path = shortestPath((d)-[*1..4]-(finding))
            WHERE finding.name IN inputs AND finding.name <> d.name
            
            // 3. Clinical Weighting (w_ti)
            WITH d, finding, path,
                 CASE 
                    WHEN finding.name CONTAINS 'Procalcitonin' THEN 15.0 // The strongest anchor
                    WHEN finding.name CONTAINS 'Fever' OR finding.name CONTAINS 'Hyperthermia' THEN 8.0
                    WHEN finding.name CONTAINS 'Chest Pain' THEN 10.0
                    WHEN 'Symptom' IN labels(finding) THEN 5.0
                    ELSE 2.0 
                 END AS w_ti
            
            // 4. Scoring Logic
            WITH d, 
                 collect(DISTINCT finding.name) AS matched_evidence,
                 sum(w_ti / (length(path)^2 + 1.0)) AS path_strength,
                 count(DISTINCT finding) AS match_count,
                 COUNT { (d)--() } AS connectivity

            // 5. Categorical Importance Boost
            WITH d, matched_evidence, path_strength, match_count, connectivity,
                 CASE 
                    WHEN d.name CONTAINS 'sepsis' OR d.name CONTAINS 'septic' THEN 8.0
                    WHEN d.name CONTAINS 'pneumonia' OR d.name CONTAINS 'infarct' THEN 6.0
                    ELSE 1.0
                 END AS category_boost

            // 6. Final Syndromic Threshold (Require at least 3 clinical links)
            WHERE match_count >= 3

            // 7. Final Math
            WITH d, matched_evidence, 
                 ((path_strength * (match_count^3)) * category_boost) / log(connectivity + 10) AS final_rank
            
            RETURN d.name AS Disease, 
                   final_rank AS FinalScore,
                   matched_evidence AS Evidence
            ORDER BY FinalScore DESC
            LIMIT 10
            """
            results = session.run(query, input_entities=entity_names)
            return [dict(r) for r in results]

class PaperDiagnosticSystem:
    def __init__(self):
        self.matcher = BioLORDMatcher()
        self.kg_agent = PaperKGAgent(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)

    def process_patient(self, raw_findings: list):
        # 1. Entity Recognition & Matching (3.2.1)
        print("🔍 Phase 3.2.1: Linking entities to KG...")
        resolved = []
        for item in raw_findings:
            match = self.matcher.match(item)
            if match:
                resolved.append({"name": match})
                print(f"  Mapped: '{item}' -> '{match}'")

        # 2. Localization & Reranking (3.2.2 & 3.2.3)
        print("\n🧠 Phase 3.2.2 & 3.2.3: KG Localization and Path Reranking...")
        candidates = self.kg_agent.generate_candidates(resolved)
        return candidates

# --- EXECUTION ---
if __name__ == "__main__":
    patient_data = [
        "High Fever",
        "WBC leukocytosis",
        "Tachycardia",
        "Chest Pain",
        "Calcitonin high"
    ]

    system = PaperDiagnosticSystem()
    try:
        results = system.process_patient(patient_data)

        print("\n" + "="*60)
        print("🧬 CANDIDATE DISEASE GENERATION (PAPER-BASED RERANKING)")
        print("="*60)
        
        for i, res in enumerate(results[:5]):
            print(f"RANK {i+1}: {res['Disease']}")
            print(f"  > Reranking Score: {round(res['FinalScore'], 4)}")
            print(f"  > Localized Evidence: {', '.join(res['Evidence'])}")
            print("-" * 30)
            
    finally:
        system.kg_agent.close()

🔍 Phase 3.2.1: Linking entities to KG...
  Mapped: 'High Fever' -> 'Hyperthermia (High Fever)'
  Mapped: 'WBC leukocytosis' -> 'Leukocytosis NOS'
  Mapped: 'Tachycardia' -> 'Tachycardia NOS'
  Mapped: 'Chest Pain' -> 'Chest Pain'
  Mapped: 'Calcitonin high' -> 'Procalcitonin high'

🧠 Phase 3.2.2 & 3.2.3: KG Localization and Path Reranking...

🧬 CANDIDATE DISEASE GENERATION (PAPER-BASED RERANKING)
RANK 1: Severe sepsis
  > Reranking Score: 1097.2123
  > Localized Evidence: Leukocytosis NOS, Tachycardia NOS, Procalcitonin high
------------------------------
RANK 2: Bacterial pneumonia NOS
  > Reranking Score: 1017.3953
  > Localized Evidence: Leukocytosis NOS, Tachycardia NOS, Procalcitonin high
------------------------------
RANK 3: Salmonella septicemia
  > Reranking Score: 782.9795
  > Localized Evidence: Leukocytosis NOS, Tachycardia NOS, Procalcitonin high
------------------------------
RANK 4: H. influenae septicemia
  > Reranking Score: 776.0887
  > Localized Evidence: Leukocytosi